# NewsBot 2.0 · Notebook 07 – System Integration & Testing
**ITAI 2373 Final Project · Mourinho Hernanto**

**Goal:** run the complete NewsBot 2.0 system end-to-end, test robustness and performance, and export reports.

**Course modules:** all modules

> Runs top-to-bottom in Google Colab (first cell installs everything). All logic lives in `src/` –
> this notebook demonstrates and evaluates it.

In [1]:
# ⚙️ Setup – works in Google Colab (clones the repo + installs) and locally
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/ITAI2373-NewsBot-Final/ITAI2373-NewsBot-Final") if IN_COLAB else Path.cwd().resolve().parent
if IN_COLAB and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Haryapandhita/ITAI2373-NewsBot-Final.git",
                    "/content/ITAI2373-NewsBot-Final"], check=True)
sys.path.insert(0, str(ROOT))
from scripts.colab_setup import setup
ROOT = setup()

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
# Transformer models (summarisation, translation, multilingual embeddings) are used when
# available (Colab: Runtime -> Change runtime type -> T4 GPU recommended); otherwise
# NewsBot falls back to classical methods automatically.
USE_TRANSFORMERS = os.environ.get("NEWSBOT_USE_TRANSFORMERS", "1") == "1"
print("Transformers enabled:", USE_TRANSFORMERS)

Project root: /home/claude/NewsBot-Final/ITAI2373-NewsBot-Final  |  Colab: False


Transformers enabled: True


In [2]:
from src.newsbot import NewsBot2
from src.data_processing.data_loader import sample_articles
bot = NewsBot2(use_transformers=USE_TRANSFORMERS).build()
pd.Series(bot.timings, name="seconds")

20:58:00 | INFO    | newsbot | load data                       0.1s


20:58:00 | INFO    | newsbot | classifier                      0.6s


20:58:00 | INFO    | newsbot | language detector               0.1s


20:58:01 | INFO    | newsbot | enrich corpus                   0.2s


20:59:53 | INFO    | src.analysis.ner_extractor | Knowledge graph: 10744 nodes, 34403 edges


20:59:53 | INFO    | newsbot | knowledge graph               112.2s


... (5 outputs left out)

21:01:28 | INFO    | newsbot | semantic index                 74.6s


21:01:28 | INFO    | newsbot | intent classifier               0.1s


21:01:28 | INFO    | newsbot | NewsBot 2.0 ready: 15242 articles, 6 languages


load data              0.06
classifier             0.55
language detector      0.09
enrich corpus          0.17
knowledge graph      112.17
topic model           20.25
summarizer             0.00
semantic index        74.64
intent classifier      0.12
Name: seconds, dtype: float64

## 1. Comprehensive analysis of one article

In [3]:
art = sample_articles()[0]
res = bot.analyze_article(art["text"], art["title"])
for k in ["language", "classification", "sentiment", "topics", "summary", "headlines"]:
    print(f"== {k}"); print(res[k])
print("== entities:", sorted({e["canonical"] for e in res["entities"]}))
print("== relations:", res["relations"]); print("== errors:", res["errors"])

== language
{'language': 'en', 'name': 'English', 'confidence': 0.9884, 'alternatives': {'it': 0.0045, 'fr': 0.0024}, 'reliable': True}
== classification
{'label': 'tech', 'confidence': 0.517, 'margin': 0.0838, 'probabilities': {'tech': 0.517, 'business': 0.4332, 'sport': 0.0232, 'politics': 0.0185, 'entertainment': 0.0081}, 'secondary_labels': ['business'], 'needs_review': True}
== sentiment
{'label': 'positive', 'score': 0.4137, 'whole_text_vader': 0.8973, 'confidence': 0.875, 'positive_sentence_share': 0.75, 'negative_sentence_share': 0.0, 'most_positive_sentence': 'Chief executive Tim Cook said the processor would make iPhones faster and more energy efficient.', 'most_negative_sentence': 'Tech giant unveils new AI chip.', 'emotions': {'anger': 0.0, 'anticipation': 0.2222, 'disgust': 0.0, 'fear': 0.2222, 'joy': 0.1111, 'sadness': 0.0, 'surprise': 0.1111, 'trust': 0.3333}, 'method': 'vader-sentence-average'}
== topics
[{'topic': 4, 'label': 'ai / technology / solutions', 'weight': 0.

In [4]:
res["enhancements"]

{'entity_background': [{'entity': 'AI',
   'type': 'GPE',
   'mentions_in_corpus': 202,
   'mostly_in': 'tech',
   'often_with': 'United States, Google, OpenAI'},
  {'entity': 'Apple',
   'type': 'ORG',
   'mentions_in_corpus': 61,
   'mostly_in': 'tech',
   'often_with': 'United States, Mac, Microsoft'}],
 'related_articles': [{'doc_id': 'webz_36ba8aa26f0c1d09',
   'title': 'Apple is opening the doors to a new era!',
   'similarity': 0.6683},
  {'doc_id': 'webz_b90c881cc58ebe75',
   'title': "This Is the Biggest Threat to Nvidia's Artificial Intelligence (AI) Dominance That Virtually No One Is Talking About - The Globe and Mail",
   'similarity': 0.6337},
  {'doc_id': 'webz_a3a88a544e57253b',
   'title': 'Rough forecast: AI to increase deadly waste by millions of tons',
   'similarity': 0.6335}],
 'tone_context': 'More positive than 96% of tech articles (category average +0.08, this article +0.41).',
 'numeric_facts': []}

## 2. A non-English article (language → translation → analysis → cultural context)

In [5]:
es = sample_articles()[3]
r = bot.analyze_article(es["text"], es["title"])
{k: r.get(k) for k in ["language", "translation", "classification", "sentiment", "cultural_context", "errors"]}

21:01:29 | WARNING | src.multilingual.translator | Translation model Helsinki-NLP/opus-mt-es-en unavailable (No module named 'transformers')


{'language': {'language': 'es',
  'name': 'Spanish',
  'confidence': 0.9984,
  'alternatives': {'pt': 0.0006, 'fr': 0.0005},
  'reliable': True},
 'translation': {'text': 'El banco central sube los tipos. El Banco Central Europeo subió los tipos de interés el jueves para frenar la inflación. La presidenta Christine Lagarde dijo que la economía sigue fuerte pero que los precios de la energía preocupan a los mercados.',
  'translated': False,
  'engine': 'unavailable'},
 'classification': {'label': 'business',
  'confidence': 0.3792,
  'margin': 0.1471,
  'probabilities': {'business': 0.3792,
   'politics': 0.232,
   'sport': 0.1892,
   'tech': 0.1202,
   'entertainment': 0.0794},
  'secondary_labels': ['politics'],
  'needs_review': True},
 'sentiment': {'label': 'neutral',
  'score': 0.0,
  'whole_text_vader': 0.0,
  'confidence': 1.0,
  'positive_sentence_share': 0.0,
  'negative_sentence_share': 0.0,
  'most_positive_sentence': 'La presidenta Christine Lagarde dijo que la economía si

## 3. Batch analysis

In [6]:
import time
batch_texts = bot.corpus.sample(40, random_state=5)["full_text"].tolist() + [a["text"] for a in sample_articles()]
t0 = time.time(); batch = bot.batch_analysis(batch_texts, progress=False)
print(f"{len(batch)} articles in {time.time()-t0:.1f}s ({1000*(time.time()-t0)/len(batch):.0f} ms/article)")
batch.head(10)

21:01:30 | WARNING | src.multilingual.translator | Translation model Helsinki-NLP/opus-mt-ROMANCE-en unavailable (No module named 'transformers')


21:01:31 | WARNING | src.multilingual.translator | Translation model Helsinki-NLP/opus-mt-fr-en unavailable (No module named 'transformers')


21:01:32 | WARNING | src.multilingual.translator | Translation model Helsinki-NLP/opus-mt-de-en unavailable (No module named 'transformers')


44 articles in 4.5s (103 ms/article)


,idx,language,category,confidence,needs_review,sentiment,sentiment_label,n_entities,top_topic,summary,errors
0,0,en,tech,0.7920,False,0.2972,positive,89,health / treatment / study,"Feng Yan , one of the creators of the video series at the nursing home, told the Global Times that the original inte...",
1,1,en,politics,0.7068,False,0.2006,positive,44,health / treatment / study,Irish Examiner view: Ireland's influence abroad helped ensure Tori Towey's safe return. Ms Towey was full of praise ...,
2,2,pt,politics,0.3550,True,-0.0470,neutral,16,cd / ed / ce,O valor de uma marca local: do boca a boca ao registro no INPI. O cliente que gostou do atendimento conta para a col...,
3,3,pt,politics,0.4006,True,-0.0505,negative,41,cd / ed / ce,O jornalista Medhat Shalabi lançou um forte ataque contra o Al-Ahly após os acontecimentos da partida contra o Ceram...,
4,4,en,business,0.8499,False,0.2205,positive,161,ai / technology / solutions,Swadeshi 1.0 vs. Made in India 2.0: How India turned a 1905 boycott into a 2026 business strategy. Data published in...,
5,5,en,business,0.8467,False,0.4561,positive,29,growth / economy / economic,The rate of the increase was not decided until December by PGE and Oregon Public Utility Commission (PUC) pending th...,
6,6,en,business,0.8119,False,0.0475,neutral,46,court / law / state,How Colorado auto and catalytic converter theft victims can get money back. Coloradans who had a vehicle or catalyti...,
7,7,en,politics,0.9386,False,0.3218,positive,32,labour / minister / party,"Rajkumar Santoshi to PM Modi on birthday: May he continue to lead us. Mumbai, Sep 17 (IANS) National Award-winning f...",
8,8,fr,politics,0.3741,True,-0.0167,neutral,39,cd / ed / ce,"Boycott du salon de l'agriculture : ""ça ne rend pas service"" pour la chambre d'agriculture de la région Centre. Une ...",
9,9,en,business,0.9466,False,-0.2390,negative,14,growth / economy / economic,"Tsunami 'to hit Sri Lanka banks'. According to its estimate, as much as 13.6% of the loans made by private banks to ...",


## 4. Robustness – edge cases

In [7]:
cases = {"empty": "", "very short": "Goal!", "numbers only": "123 456 789", "gibberish": "asdkj qwpoe zmxnc",
         "very long": " ".join([sample_articles()[1]["text"]] * 60), "mixed language": "Apple sube en bolsa. Shares rallied in New York."}
for name, text in cases.items():
    try:
        r = bot.analyze_article(text, enhance=False)
        c = r.get("classification", {})
        print(f"{name:15} → {c.get('label')} (review: {c.get('needs_review')}) | lang {r.get('language', {}).get('language')} "
              f"| warnings: {r.get('warnings', [])} | errors: {r['errors']}")
    except ValueError as e:
        print(f"{name:15} → rejected cleanly: {e}")

empty           → rejected cleanly: Empty article
very short      → sport (review: True) | lang und | warnings: ['input too short or language undetermined (1 words, language=und) – results are unreliable'] | errors: {}
numbers only    → sport (review: True) | lang und | warnings: ['input too short or language undetermined (3 words, language=und) – results are unreliable'] | errors: {}
gibberish       → sport (review: True) | lang und | warnings: ['input too short or language undetermined (3 words, language=und) – results are unreliable'] | errors: {}


very long       → politics (review: False) | lang en | warnings: [] | errors: {}
mixed language  → business (review: True) | lang en | warnings: [] | errors: {}


## 5. Insights report and export

In [8]:
report = bot.generate_insights_report()
print(report[:2500])

# NewsBot 2.0 – Insights report

*15,242 articles · 6 languages · report type: comprehensive*

## Key insights

- 15,242 articles analysed; largest category is politics (30% of coverage).
- Most negative coverage: politics (+0.01); most positive: entertainment (+0.18).
- Tone differs by language: de sources are the most negative (-0.21), en the most positive (+0.10).
- Overall tone worsened from 2023Q4 (+0.09) to 2026Q4 (+0.05).
- Most mentioned entities: United States (2002), United Kingdom (833), Donald Trump (663), China (432), AI (420).

## Sentiment by category

| category      |   mean |   count |
|:--------------|-------:|--------:|
| business      |  0.098 |    2677 |
| entertainment |  0.18  |     366 |
| politics      |  0.013 |    4593 |
| sport         |  0.062 |    4116 |
| tech          |  0.085 |    3490 |

## Coverage by language

| language   |   business |   entertainment |   politics |   sport |   tech |
|:-----------|-----------:|----------------:|-----------:|-----

In [9]:
from src.utils.export import export_results
out = bot.cfg.results_dir / "exports"
print(export_results(report, out / "insights_report.html"))
print(bot.export_article_report(res, out / "article_report.md"))
print(export_results(batch, out / "batch_analysis.csv"))

/home/claude/NewsBot-Final/ITAI2373-NewsBot-Final/data/results/exports/insights_report.html
/home/claude/NewsBot-Final/ITAI2373-NewsBot-Final/data/results/exports/article_report.md
/home/claude/NewsBot-Final/ITAI2373-NewsBot-Final/data/results/exports/batch_analysis.csv


## 6. Automated tests

In [10]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True, cwd=ROOT).stdout[-1500:])

.............................                                            [100%]
=============================== warnings summary ===============================
tests/test_classification.py::test_evaluate_keys
  /usr/local/lib/python3.13/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
29 passed, 1 warning in 110.31s (0:01:50)



## 7. Results of the full experiment run (`scripts/run_experiments.py`)

In [11]:
from src.utils.evaluation import load_json
M = load_json(ROOT / "data/results/metrics.json")
summary = {
  "BBC test accuracy (ensemble)": M["classification"]["ensemble_bbc_test"]["accuracy"],
  "BBC test macro-F1 (ensemble)": M["classification"]["ensemble_bbc_test"]["macro_f1"],
  "2023-26 news accuracy, BBC-only → NewsBot 2.0": f'{M["classification"]["cross_domain_webz_en"]["bbc_only_model"]["accuracy"]} → {M["classification"]["cross_domain_webz_en"]["newsbot2_model"]["accuracy"]}',
  "Sentiment agreement with Webz labels": M["sentiment"]["accuracy_sentence_avg_news_lexicon"],
  "Topic NPMI coherence (NMF k=15)": M["topics"]["final"]["npmi"],
  "Language ID accuracy (first 20 chars)": M["language_detection"]["first_20_chars"]["accuracy"],
  "Intent accuracy (held-out)": M["conversation"]["held_out_accuracy"],
  "Search precision@10": M["semantic_search"]["retrieval_precision_at_10"]["precision_at_k"],
}
pd.Series(summary, name="value")

BBC test accuracy (ensemble)                              0.9737
BBC test macro-F1 (ensemble)                              0.9727
2023-26 news accuracy, BBC-only → NewsBot 2.0    0.6875 → 0.8296
Sentiment agreement with Webz labels                      0.6993
Topic NPMI coherence (NMF k=15)                           0.4254
Language ID accuracy (first 20 chars)                     0.7933
Intent accuracy (held-out)                                0.9767
Search precision@10                                       0.7823
Name: value, dtype: object

## Conclusion
All four modules run together through one orchestrator (`NewsBot2`): any supported-language article is detected, translated
when needed, classified with calibrated confidence, scored for sentiment and emotions, mined for entities/relations, placed
in topics, summarised, enriched with grounded context and made explorable through a context-aware chatbot. Every step is
isolated so one failure never breaks the whole analysis, and heavy artefacts are cached for fast restarts.